# Konversi Model Keras/TensorFlow ke ONNX di Google Colab

Notebook ini digunakan untuk mengonversi model **Typing Stress** (`.h5`) ke format **ONNX** (`.onnx`) secara cepat menggunakan Google Colab.

### Langkah-langkah:
1. Klik ikon folder di sidebar sebelah kiri Colab.
2. Unggah file model `typing_stress_lstm.h5` dari komputer Anda.
3. Jalankan sel kode di bawah ini.
4. Setelah selesai, unduh file `typing_stress_lstm.onnx` yang terbentuk dan masukkan kembali ke folder `models/typing_model/` proyek Anda.

In [ ]:
# 1. Install tf2onnx library
!pip install tf2onnx

In [ ]:
# 2. Kode konversi model dengan custom layers
import os
import tensorflow as tf
import tf2onnx

# Custom compatibility layers agar Keras bisa membaca model dengan benar
class NotEqual(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(NotEqual, self).__init__(**kwargs)
    def __call__(self, *args, **kwargs):
        tensor_input = args[0]
        return super(NotEqual, self).__call__(tensor_input, **kwargs)
    def call(self, x):
        return tf.math.not_equal(x, 0.0)

class Any(tf.keras.layers.Layer):
    def __init__(self, axis=-1, keepdims=False, **kwargs):
        super(Any, self).__init__(**kwargs)
        self.axis = axis
        self.keepdims = keepdims
    def call(self, x):
        return tf.reduce_any(x, axis=self.axis, keepdims=self.keepdims)
    def get_config(self):
        config = super(Any, self).get_config()
        config.update({'axis': self.axis, 'keepdims': self.keepdims})
        return config

class PatchedDense(tf.keras.layers.Dense):
    def __init__(self, *args, **kwargs):
        kwargs.pop('quantization_config', None)
        super(PatchedDense, self).__init__(*args, **kwargs)

class PatchedLSTM(tf.keras.layers.LSTM):
    def __init__(self, *args, **kwargs):
        kwargs.pop('quantization_config', None)
        super(PatchedLSTM, self).__init__(*args, **kwargs)

class PatchedMasking(tf.keras.layers.Masking):
    def __init__(self, *args, **kwargs):
        kwargs.pop('quantization_config', None)
        super(PatchedMasking, self).__init__(*args, **kwargs)

CUSTOM_OBJECTS = {
    'NotEqual': NotEqual,
    'Any': Any,
    'Dense': PatchedDense,
    'LSTM': PatchedLSTM,
    'Masking': PatchedMasking
}

model_path = "typing_stress_lstm.h5"
output_path = "typing_stress_lstm.onnx"

if not os.path.exists(model_path):
    print(f"❌ ERROR: File '{model_path}' belum di-upload ke Colab. Silakan upload terlebih dahulu!")
else:
    print(f"🔄 Memuat model Keras: {model_path}...")
    model = tf.keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS)
    
    print("🔄 Mengonversi model ke format ONNX...")
    spec = (
        tf.TensorSpec((None, 50, 1), tf.float32, name="seq_input"),
        tf.TensorSpec((None, 3), tf.float32, name="static_input"),
    )
    
    model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13)
    
    print(f"💾 Menyimpan model ONNX ke: {output_path}...")
    with open(output_path, "wb") as f:
        f.write(model_proto.SerializeToString())
        
    print("✅ SELESAI! Silakan download file 'typing_stress_lstm.onnx' dari panel sebelah kiri.")